Bike Sharing (UCI)

Projet fil rouge : prédire et comprendre la demande de vélos en libre-service à partir de variables temporelles et météo.

Ce capstone combine :
- **Régression** : prédire `cnt` (demande horaire)
- **Classification** : détecter les *pics* de demande (`is_peak`)
- **Arbres / Forêts / Ensembles** : Random Forest, Extra Trees, etc.
- **Boosting** : HistGradientBoosting (ou autre) + (optionnel) XGBoost / LightGBM / CatBoost
- **Clustering** : segmentation de profils de journées (courbe 24h)
- **Réduction de dimension** : PCA / t-SNE / UMAP (optionnel) pour visualisation
- **Communication** : mini-rapport + livrables (modèle, config)

Principe important : **éviter la fuite temporelle** → split chronologique / backtesting simple.


## Données (téléchargement fiable)

Dataset : **Bike Sharing Dataset** (Capital Bikeshare, Washington D.C., 2011–2012).

- Zip officiel (UCI) : `Bike-Sharing-Dataset.zip`
- Fichiers : `hour.csv` (données horaires) et `day.csv` (données journalières)

Le code ci-dessous :
1) télécharge le zip depuis UCI
2) extrait `hour.csv` et `day.csv` dans `./data/`


In [ ]:
import zipfile
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ZIP_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip"
ZIP_PATH = DATA_DIR / "Bike-Sharing-Dataset.zip"

def download(url: str, dest: Path, chunk_size: int = 1 << 20):
    import requests
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)

if not ZIP_PATH.exists():
    print("Downloading:", ZIP_URL)
    download(ZIP_URL, ZIP_PATH)
else:
    print("Zip already exists:", ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extract("hour.csv", path=DATA_DIR)
    z.extract("day.csv", path=DATA_DIR)

hour_path = DATA_DIR / "hour.csv"
day_path = DATA_DIR / "day.csv"
hour_path, day_path

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

## 0) Lecture rapide (contexte & variables)

Avant de coder, lire `Readme.txt` du zip (optionnel) et repérer :
- la variable cible `cnt`
- les variables temporelles (`dteday`, `hr`, `weekday`, `workingday`, `season`, `yr`, `mnth`)
- les variables météo (`temp`, `atemp`, `hum`, `windspeed`, `weathersit`)
- les variables potentiellement problématiques pour la fuite (`casual`, `registered`)

👉 Consigne : dans ce projet, **ne pas utiliser** `casual` et `registered` comme features pour prédire `cnt` (ce sont des composantes de `cnt`).


## 1) Chargement des données

À faire :
1. Charger `hour.csv` dans `df_hour` et `day.csv` dans `df_day`.
2. Afficher :
   - `shape`
   - `head()`
   - `info()`
3. Vérifier rapidement :
   - valeurs manquantes
   - doublons (s'il y en a)
   - distributions de `cnt`

Livrable :
- 3–5 lignes Markdown résumant ce que contiennent les deux tables et ce que tu vas utiliser.


In [ ]:
# À compléter
# df_hour = pd.read_csv(...)
# df_day  = pd.read_csv(...)

## 2) EDA orientée décision (sur `df_hour`)

Objectif : répondre à “qu'est-ce qui explique la demande ?”

À faire :
1) **Temps**
- convertir `dteday` en datetime
- tracer `cnt` en série temporelle (agrégé par jour, puis par semaine si utile)
- tracer le profil horaire moyen : `mean(cnt)` par `hr`
- tracer `mean(cnt)` par `weekday` et `workingday`

2) **Saisonnalité**
- `mean(cnt)` par `season` et par `mnth`

3) **Météo**
- scatter `temp` vs `cnt` (éventuellement en ajoutant une tendance)
- boxplot de `cnt` par `weathersit`
- effet `hum` et `windspeed` (scatter / bins)

4) **Checks**
- vérifier les plages (0–1 pour variables normalisées)
- repérer des points extrêmes (jours/horaires anormaux)

Questions à répondre (en markdown) :
- à quelles heures les pics apparaissent-ils ?
- le weekend change-t-il le profil ?
- quelle variable météo semble la plus influente ?


In [ ]:
# À compléter : plots + stats

## 3) Split temporel (anti-fuite)

Pourquoi : si on split au hasard, on “mélange” le futur dans le train → résultat trop optimiste.

À faire :
1) Trier `df_hour` par (`dteday`, `hr`)
2) Split chronologique :
   - train = premières 80% lignes (dans le temps)
   - test  = dernières 20% lignes
3) Définir :
   - `y = cnt`
   - `X` = toutes les features **sauf** `cnt`, `casual`, `registered`
4) Vérifier qu'il n'y a pas de fuite :
   - pas de colonnes post-cible
   - pas de mélange temporel

Livrable :
- afficher la date min/max train et test.


In [ ]:
# À compléter

## 4) Feature engineering

### 4.1 Variables cycliques

Les variables “circulaires” (heure, mois) ne sont pas bien représentées comme des entiers :
- 23h et 0h sont proches, mais numériquement 23 et 0 sont loin.
On encode donc en sin/cos.

À faire :
- créer `hr_sin`, `hr_cos`
- créer `mnth_sin`, `mnth_cos`

### 4.2 Interactions simples

À faire (au moins 2) :
- `temp_x_wind = temp * windspeed`
- `work_x_hr = workingday * hr` (ou *hr_sin* selon ton choix)

### 4.3 (option) Transform cible

- créer `log_cnt = log1p(cnt)` et tester si cela stabilise les erreurs
- attention : il faut inverser `expm1` pour interpréter en “cnt”.

Livrable :
- expliquer en markdown pourquoi ces features peuvent aider.


In [ ]:
# À compléter

## 5) Prétraitement : catégorielles vs numériques (bonnes pratiques)

Objectif : préparer un pipeline robuste et reproductible.

À faire :
1) Identifier les colonnes catégorielles (faible cardinalité) :
   - `season`, `yr`, `holiday`, `weekday`, `workingday`, `weathersit`
2) Définir les colonnes numériques = le reste
3) Construire un `ColumnTransformer` :
   - numériques : imputation median + scaler
   - catégorielles : imputation mode + OneHotEncoder

Livrable :
- afficher le nombre de features après OneHot (option : récupérer les noms).


In [ ]:
# À compléter

## 6) Régression : baselines

À faire :
1) `DummyRegressor` (moyenne) → baseline minimal
2) `Ridge` (avec pipeline) → baseline “linéaire régularisée”
3) Évaluer sur test :
   - MAE
   - RMSE
   - R²
4) Produire un tableau comparatif trié par RMSE.

Livrable :
- 5 lignes max de commentaire (quel modèle est déjà “utile” ?).


In [ ]:
# À compléter

## 7) Arbres / forêts / ensembles

### Random Forest
Ensemble d'arbres entraînés sur des bootstraps + sous-ensemble de variables à chaque split.
➡️ Capture des non-linéarités et réduit la variance.

Hyperparamètres clés :
- `n_estimators` : nb d'arbres
- `max_depth` : profondeur (contrôle sur-apprentissage)
- `min_samples_leaf` : taille min feuille (régularisation)
- `max_features` : nb de variables testées par split

### ExtraTrees (intro)
Similaire à RandomForest mais splits plus randomisés.
➡️ Souvent robuste, parfois meilleur, parfois plus biaisé.

À faire :
1) Entraîner `RandomForestRegressor`
2) Entraîner `ExtraTreesRegressor`
3) Comparer au meilleur baseline
4) Interprétation :
   - top-15 importances
   - 3 features “inattendues” à discuter

Bonus :
- mini grille sur `max_depth` / `min_samples_leaf`.


In [ ]:
# À compléter

## 8) Boosting

Le boosting ajoute des modèles faibles **séquentiellement**, chaque étape corrige une partie des erreurs.
➡️ Très performant mais sensible à la régularisation.

À faire :
1) `HistGradientBoostingRegressor` (ou autres)
2) Optionnel : XGBoost / LightGBM / CatBoost
   - utiliser `try/except import`
   - si non dispo : message et continuer

Évaluer et comparer.

Livrable :
- expliquer l’effet de `learning_rate`, `max_depth` (ou `num_leaves`), `n_estimators/max_iter`.


In [ ]:
# À compléter

## 9) Classification : prédire les pics (is_peak)

Transformation : au lieu de prédire `cnt`, on veut détecter les périodes de forte demande.

À faire :
1) Définir `is_peak` :
   - seuil = quantile 0.85 de `cnt` **sur train seulement**
2) Modèles à comparer :
   - LogisticRegression (avec pipeline)
   - RandomForestClassifier
   - (option) HistGradientBoostingClassifier
3) Métriques :
   - F1
   - Recall
   - PR-AUC (recommandée si classe rare)
4) (bonus) seuil de décision :
   - choisir un seuil garantissant precision >= 0.80 et maximiser recall.

Livrable :
- expliquer le compromis precision/recall selon le contexte (ex : repositionnement de vélos).


In [ ]:
# À compléter

## 10) Clustering : segmentation de profils journaliers

Idée : créer un “profil journée” = courbe 24h de la demande.

À faire :
1) Construire `day_profile` :
   - pivot : index = `dteday`, colonnes = `hr`, valeurs = `mean(cnt)`
2) Standardiser les 24 colonnes
3) KMeans :
   - tester k=3..10
   - silhouette
4) Interpréter :
   - tracer la courbe moyenne 24h par cluster
   - relier clusters à `workingday`, `season`, `weathersit` (stats par cluster)
5) (option) DBSCAN :
   - tester 2–3 configs et comparer

Livrable :
- donner un nom “métier” à chaque cluster.


In [ ]:
# À compléter

## 11) Réduction de dimension : PCA / t-SNE / UMAP

Objectif : visualiser `day_profile` en 2D et voir si les clusters se séparent.

- **PCA** : projection linéaire (variance expliquée)
- **t-SNE** : visualisation non-linéaire des voisinages (perplexity)
- **UMAP** : alternative non-linéaire (optionnelle si `umap-learn`)

À faire :
1) PCA 2D, coloré par cluster
2) t-SNE 2D (perplexity 20/40), coloré par cluster
3) UMAP 2D (option), coloré par cluster
4) Commenter les limites : “visualisation ≠ preuve”.

Livrable :
- 5 lignes sur ce que tu observes.


In [ ]:
# À compléter

## 12) Communication & livrables

Créer un dossier `deliverables/` contenant :
- `report.md` (structure suggérée)
- `best_model.joblib`
- `config.json`

`report.md` doit inclure :
1) Contexte & données
2) 3 insights EDA
3) Résultats régression (tableau + justification)
4) Résultats classification (métriques + seuil si fait)
5) Segmentation (clusters) + usages
6) Recommandations (3 bullets)
7) Limites & prochaines étapes (3 bullets)

Bonus :
- ajouter une section “risques de déploiement” (drift, saisonnalité, monitoring).


In [ ]:
# À compléter